# មេរៀនទី ១០ - យន្តករបញ្ញាសិប្បនិម្មិតនៅក្នុងការផលិត

ក្នុងមេរៀននេះ អ្នកនឹងរៀនអំពី **លំនាំផលិតកម្ម** សម្រាប់យន្តករបញ្ញាសិប្បនិម្មិតដោយប្រើ Microsoft Agent Framework (Python)។ យើងបានគ្របដណ្តប់:

- **ការត្រួតពិនិត្យមើល** — ការបន្ថែមពេលវេលា និងកំណត់ហេតុទៅលើប្រតិបត្តិការយន្តករ
- **ការវាយតម្លៃ** — ការប្រើយន្តករវាយតម្លៃដើម្បីផ្ដល់ពិន្ទុគុណភាពចម្លើយ
- **ការគ្រប់គ្រងថ្លៃឈ្នួល** — គម្រោងសម្រាប់កំណត់ផ្ទះសំណាយនិងជ្រើសរើសម៉ូដែល

ស្ថានភាពនេះគឺជារ៉ូបូសេវាធ្វើការធ្វើដំណើរដើម្បីជួយអ្នកប្រើរៀបចំដំណើរកំសាន្ត ជាមួយនឹងការត្រួតពិនិត្យ និងការវាយតម្លៃដាក់លើកំពូល។


## ការត្រូវចែងតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ការពិចារណាសម្រាប់ការផលិត

ការផ្លាស់ប្តូរអ្នកប្រើប្រាស់ AI ពីគំរូដើមទៅការផលិតត្រូវការយកចិត្តទុកដាក់យ៉ាងម៉តចតជាមួយស្ថូរក្នុងបីយ៉ាង:

1. **ការសង្កេតមើល** — អ្នកត្រូវការមើលឃើញអំពីអ្វីដែលភ្នាក់ងារធ្វើ, រយៈពេលវាទៅលើការធ្វើ, និងឧបករណ៍ណាដែលវាមកហៅ។ រីលរឺប្រព័ន្ធការចុះបញ្ជីដោយគ្មាន ការតាមដាននិងកំណត់ហេតុ, ការពិនិត្យកំហុសក្នុងការផលិតគឺជារឿងជាងតែតាមដានមិនឃើញ។

2. **ការវាយតម្លៃ** — ការត្រួតពិនិត្យគុណភាពដោយស្វ័យប្រវត្តិធានាថា ឆ្លើយតបរបស់ភ្នាក់ងារទៅលើការពិត, ពេញលេញ និងមានប្រយោជន៍នៅពេលវេលាអនាគត។ ភ្នាក់ងារវាយតម្លៃអាចផ្ដល់ពិន្ទុឆ្លើយតបទៅលើលក្ខខណ្ឌដែលបានកំណត់។

3. **ការគ្រប់គ្រងការចំណាយ** — ការប្រើប្រាស់សញ្ញាតំណាងផ្ទាល់ប៉ះពាល់ទៅលើការចំណាយ។យុទ្ធសាស្ត្រដូចជា ការបង្កើតសំណើអតិបរមា, ការជ្រើសរើសម៉ូដែល និងការរក្សាទុកអាចជួយរក្សាការចំណាយឱ្យនៅក្រោមការត្រួតពិនិត្យដោយគ្មានបាត់បង់គុណភាព។


## ការជួញដូរ​អេហ្សង់ Observable

យើងបានកំណត់ឧបករណ៍ធ្វើដំណើរ និងបង្ហាប់ការហៅ​អេហ្សង់ជាមួយពេលវេលា ដូច្នេះយើងអាចតាមដានស្ពត់​បាន។ នៅក្នុង​ផលិតកម្ម អ្នកនឹងត្រូវបញ្ចូលជាមួយ OpenTelemetry ឬសេវាបញ្ជាម្មីដូចគ្នា។


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## ម៉ូដលំនាំការវាយតម្លៃ

លំនាំផលិតកម្មទូទៅគឺការប្រើភ្នាក់ងារទីពីរ​ជា **អ្នកវាយតម្លៃ**។ អ្នកវាយតម្លៃវាយតម្លៃចម្លើយរបស់ភ្នាក់ងារចម្បងផ្ទៀងផ្ទាត់ទៅនឹងគោលការណ៍កំណត់ដូចជា ភាពពេញលេញ, ភាពត្រឹមត្រូវ, និងភាពជួយសម្រួល។

នេះអនុញ្ញាតឱ្យមាន៖
- ទ្វារខុសត្រូវគុណភាពដោយស្វ័យប្រវត្តិមុនពេលចម្លើយឈានដល់អ្នកប្រើ
- ការធ្វើសេចក្តីបញ្ជាក់វិលត្រឡប់ពេលមានការផ្លាស់ប្តូរជំនួញឬម៉ូឌែល
- ការត្រួតពិនិត្យបន្តរៀងរាល់ការប្រតិបត្ដិការភ្នាក់ងារជាថ្មីៗ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## កិច្ចយុទ្ធសាស្ត្រគ្រប់គ្រងការចំណាយ

ការគ្រប់គ្រងការចំណាយគឺសំខាន់សម្រាប់តំណាង AI ផលិតកម្ម។ នេះគឺជាយុទ្ធសាស្ត្រសំខាន់ៗ៖

| វិធីសាស្ត្រ | ការពិពណ៌នា |
|---|---|
| **ការបញ្ចូលបញ្ជាឲ្យល្អប្រសើរ** | រក្សាពាក្យបញ្ជារបស់ប្រព័ន្ធអោយខ្លី។ ដកស្រង់បរិបទពុំចាំបាច់ដើម្បីបន្ថយកម្រិតសញ្ញាអញ្ញើញ។ |
    "| **ការជ្រើសរើសម៉ូដែល** | ប្រើម៉ូដែលតូច និងថ្លៃថោក (ឧ. GPT-5-mini) សម្រាប់កិច្ចការងារងាយៗដូចជាការបែងចែកឬការដកស្រង់ ហើយរក្សាម៉ូដែលធំសម្រាប់ការគិតគូរលំបាក។ |\n",
| **កាវស៊ីង** | រក្សារកម្មវិធីនិងសំណួរជាដើមជាទម្រង់កែវស៊ី ដើម្បីចៀសវាងការហៅ API មិនចាំបាច់។ |
| **ថវិកាសញ្ញា** | កំណត់ដែនកំណត់ `max_tokens` ដើម្បីទប់ស្កាត់ចម្លើយវែងខ្លាំងមិនបានរំពឹងទុក។ |
| **ក្រុមតំណាង** | ជួរប្រមូលសំណួរច្រើនរបស់អ្នកប្រើទៅក្នុងការហៅ API តែមួយ ប្រសិនបើធ្វើអាចទៅរួច។ |

ក្នុងការអនុវត្ត វិធីសាស្ត្រជាជាន់ដំណាក់កាលមានប្រសិទ្ធិភាព៖ ផ្ញើសំណើងាយៗទៅម៉ូដែលដែលរហ័ស និងថ្លៃថោក ហើយកែលម្អសំណួរលំបាកទៅម៉ូដែលមានសមត្ថភាពខ្ពស់ជាង។


## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនរបៀប:

1. **បន្ថែមការបង្កេីតចំនុចសម្រាប់មើលเห็น** ការប្រាស្រ័យទាក់ទងរបស់តំណាងជាមួយពេលវេលា និងកំណត់ហេតុ ដើម្បីដាក់ដំណាក់កាលសម្រាប់ការតាមដាន និងការត្រួតពិនិត្យ។
2. **ប៉ាន់ប្រមាណចម្លើយរបស់តំណាង** ដោយស្វ័យប្រវត្តិប្រើតំណាងវាយតម្លៃ ដែលវាយតម្លៃភាពពេញលេញ ភាពត្រឹមត្រូវ និងភាពជួយសម្រួល។
3. **គ្រប់គ្រងការចំណាយ** តាមរយៈការបង្កើនប្រសិទ្ធភាពពាក្យបញ្ជា ការជ្រើសរើសម៉ូដែល ការផ្ទុកទិន្នន័យជាមុន និងថវិកាតួអក្សរ។

របៀបផលិតកម្មទាំងនេះជួយធានាថាតំណាង AI របស់អ្នកមានភាពទៀងទាត់ អាចវាស់វែងបាន និងមានទំហំនឹងមានផលប្រយោជន៍សមរម្យ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
